In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [15]:
load = pd.read_parquet("data/processed/carga/SE_cargaverificada.parquet")

# CHANGE UTC TO BRT (UTC -03h)
load["din_referenciautc"] = (
    pd.to_datetime(load["din_referenciautc"], utc=True)
    .dt.tz_convert("America/Sao_Paulo")
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)
load.insert(0, "din_referencia", load.pop("din_referenciautc"))  # CHANGE ORDER
load.head()

,din_referencia,cod_areacarga,din_atualizacao,dat_referencia,val_cargaglobal,val_cargaglobalcons,val_cargaglobalsmmgd,val_cargasupervisionada,val_carganaosupervisionada,val_cargammgd,val_consistencia
0,2023-01-01 00:30:00,SE,2025-06-26T03:09:09.259Z,2023-01-01,451.23654,451.23654,451.23654,431.23932,19.997223,0.0,0
1,2023-01-01 01:00:00,SE,2025-06-26T03:09:09.259Z,2023-01-01,450.42630,450.42630,450.42630,430.42630,20.000000,0.0,0
2,2023-01-01 01:30:00,SE,2025-06-26T03:09:09.259Z,2023-01-01,446.17953,446.17953,446.17953,427.17900,19.000555,0.0,0
3,2023-01-01 02:00:00,SE,2025-06-26T03:09:09.259Z,2023-01-01,449.39465,449.39465,449.39465,430.39465,19.000000,0.0,0
4,2023-01-01 02:30:00,SE,2025-06-26T03:09:09.259Z,2023-01-01,450.63200,450.63200,450.63200,432.63144,18.000555,0.0,0


In [16]:
from plotly.subplots import make_subplots

plot_load = load.sort_values("din_referencia").copy()
plot_load["din_referencia"] = pd.to_datetime(plot_load["din_referencia"])

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.2,
    subplot_titles=(
        "Carga Global sem MMGD e Carga MMGD",
        "Carga Supervisionada e Nao Supervisionada",
    ),
)

fig.add_scatter(
    x=plot_load["din_referencia"],
    y=plot_load["val_cargaglobalcons"],
    mode="lines",
    name="Carga Global sem MMGD",
    row=1,
    col=1,
)
fig.add_scatter(
    x=plot_load["din_referencia"],
    y=plot_load["val_cargammgd"],
    mode="lines",
    name="Carga MMGD",
    row=1,
    col=1,
)
fig.add_scatter(
    x=plot_load["din_referencia"],
    y=plot_load["val_cargasupervisionada"],
    mode="lines",
    name="Carga Supervisionada",
    row=2,
    col=1,
)
fig.add_scatter(
    x=plot_load["din_referencia"],
    y=plot_load["val_carganaosupervisionada"],
    mode="lines",
    name="Carga Nao Supervisionada",
    row=2,
    col=1,
)

fig.update_layout(
    height=700,
    title="Carga do Nordeste",
    hovermode="x unified",
)
fig.update_yaxes(title_text="MW", row=1, col=1)
fig.update_yaxes(title_text="MW", row=2, col=1)
fig.update_xaxes(
    title_text="Data",
    row=2,
    col=1,
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=[
            dict(count=7, label="1 semana", step="day", stepmode="backward"),
            dict(count=1, label="1 mes", step="month", stepmode="backward"),
            dict(count=6, label="6 meses", step="month", stepmode="backward"),
            dict(count=1, label="1 ano", step="year", stepmode="backward"),
            dict(label="Tudo", step="all"),
        ]
    ),
)
fig.show()